# Using PyTorch Geometric

You will need to install the pyg package. With pytorch 1.8.0 or newer you can simply use:

```conda install pyg -c pyg -c conda-forge```

$$ \newcommand{\myvec}[1]{\mathbf{#1}}
\newcommand{\mymat}[1]{\mathbf{#1}} $$

In [1]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool

We start by defining a simple graph in PyTorch Geometric. For illustration, we define node attributes as 2D vectors, and edge attributes as scalars, packed into 2D tensors as shown.

The graph connectivity is specified in *Coordinate list* (COO) format - a $2\times N^e$ tensor with a column for each edge giving the sender and receiver node indices. Consider the following graph from this unit, which has been revised to use indices from zero.

![simple graph](reindexed.svg)

In [3]:
# Edges go from 2->0, 0->1, 0->1, etc.
# Tensor expects "columns" (2XN vectors), with sender in one vector and receiver in the other
# Matching indices indicate that the edge connects the sender to receiver
# edge_index tensor defines the graph
edge_index = torch.tensor([[2, 0, 0, 2, 0, 1],
                           [0, 1, 1, 1, 0, 3]], dtype=torch.long)

# x specifies the value attributes of the nodes
# this specific examples specifies that each node has a 2D attribute (usually it is 32 or 64 features)
x = torch.tensor([[1,2], [3,2], [2,2], [3,2]], dtype=torch.float)

# edge_attr specifies a scalar attribute
edge_attr = torch.tensor([[2], [4], [2], [3], [1], [5]], dtype=torch.float)

data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
print(data)

# Returns: Data(x=[4, 2], edge_index=[2, 6], edge_attr=[6, 1])

Data(x=[4, 2], edge_index=[2, 6], edge_attr=[6, 1])



In PyTorch.Geometric, a general mechanism for updating node attribute values is defined by the base class [MessagePassing](https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html#torch_geometric.nn.conv.message_passing.MessagePassing).

This base class assumes the following general form for the update of node attribute values:

$$ \myvec{v}'_i=\phi^v(\rho^{e->v}(\{\phi^e(\myvec{e}_k,\myvec{v}_{r_k},\myvec{v}_{s_k})\}_{r_k=i,k=1:N^e}),\myvec{v}_i) $$

This is effectively steps 1-3 of the framework due to Battaglia et al., with the important difference that 
the edge attribute values have scalar values and are not updated, and no use is made of a global attribute. Only the node attribute values are updated.

PyTorch Geometric provides many specialisations of the base message-passing class.  Prominent in these is [GCNConv](https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html#torch_geometric.nn.conv.GCNConv\), which defines the functions $\phi^v, \rho^{e->v}$ and $\phi^e$ (from steps 1-3) to provide the following simple update to each node attribute value:

$$ \myvec{v}'_i = \mymat{W} \sum_{r_k=i,k=1:N^e} \frac{e_k}{\lambda_{i{s_k}}}\myvec{v}_{s_k} $$

Where $\lambda_{ij}$ is the geometric mean of the respective sums of the edge attribute values received by nodes $i$ and $j$ (Geometric Mean multiples all $n$-terms and takes the $n$-root of it, vs the Arithmetic Mean which adds up $n$-terms and divides by $n$). Thus,

$$\phi^v(\myvec{x})=\mymat{W}\myvec{x}$$

$$\rho^{e->v}(A) = \textit{sum of the elements in} A$$

$$\phi^e(e_k,\myvec{v}_{r_k},\myvec{v}_{s_k}) = \frac{e_k}{\lambda_{{r_k}{s_k}}}\myvec{v}_{s_k}$$

In the following model, there are three iterations of GCNConv (i.e. three GN Blocks).

Finally, we find the mean value of all node attributes (Step 5), then apply a linear classifier followed by softmax (Step 6). Step 4 is not needed.

PyTorch.Geometric processes data in batches as for `nn.Modules` but instead of composing these along an additional tensor dimension, we treat the example graphs in a batch as a single graph with the examples as disconnected subgraphs. The only housekeeping needed is to provide an integer vector giving the graph index for every node.

In [4]:
class GCN(torch.nn.Module):
    def __init__(self, num_features, num_classes):
        super(GCN, self).__init__()
        torch.manual_seed(1234)
        self.conv1 = GCNConv(num_features, 4)
        self.conv2 = GCNConv(4, 4)
        self.conv3 = GCNConv(4, 2)
        self.lin = nn.Linear(2, num_classes)

    def forward(self, x, edge_index, edge_weight, batch):
        
        h = self.conv1(x, edge_index, edge_weight)
        h = h.relu()
        h = self.conv2(h, edge_index, edge_weight)
        h = h.relu()
        h = self.conv3(h, edge_index, edge_weight)  # Final GNN embedding space for node attributes.
       
        # Readout layer
        x = global_mean_pool(h, batch)  # [batch_size, num_features]
        x = self.lin(x)    # Apply a final classifier to each row (i.e. to the mean attribute value from each example)
        x = x.softmax(1)   # Apply softmax to each row (i.e. along dimension 1)
        
        return x

model = GCN(data.num_features,2)
print(model)

# we have a single graph, so every node has graph index 0
batch = torch.zeros((x.size()[0]),dtype=torch.long)

# apply model to our graph
model(data.x, data.edge_index, data.edge_attr.squeeze(), batch)

# Output is: tensor([[0.7365, 0.2635]], grad_fn=<SoftmaxBackward0>)
# which returns probability of the graph belonging to the two classes

GCN(
  (conv1): GCNConv(2, 4)
  (conv2): GCNConv(4, 4)
  (conv3): GCNConv(4, 2)
  (lin): Linear(in_features=2, out_features=2, bias=True)
)


tensor([[0.7365, 0.2635]], grad_fn=<SoftmaxBackward0>)

The only thing remaining is to train our network on a more realistic dataset for which we also have the ground-truth class for each graph. 

PyTorch.Geometric provides a number of Colab notebooks to illustrate its use.  [One of these](https://colab.research.google.com/drive/1I8a0DfQ3fI7Njc62__mVXUlcAleUclnb?usp=sharing) is concerned with classifying graphs.  Have a look at this notebook and follow the code through. The example uses a dataset of graphs representing molecular structures and ground-truth properties. You will see that the loss is simply the cross-entropy obtained from the output distributions associated with each graph and the ground-truth class, essentially the same as we've used in earlier units. There is further explanation of the way in which batches are handled. Note that the notebook doesn't use edge attributes in the GNN.

In [1]:
import numpy as np

In [4]:
A = np.array([[0,1,1],[0,0,1],[0,0,0]])
N = np.array([7,8,9]).T
print(A)
print(N)
print(A.dot(N))

[[0 1 1]
 [0 0 1]
 [0 0 0]]
[7 8 9]
[17  9  0]
